# Part 2: Data Preparation

**Course:** 2026 KMITL Data Analytics

A model learns from the data we give it, so the data must be correct, complete
and on a similar scale. This notebook builds a small customer dataset, finds its
problems, cleans it step by step, prepares numbers and categories, splits it,
and scales the features without leaking information.

## Learning objectives

By the end of this notebook you can:

1. Explore a dataset: shape, columns, data types, summary statistics and target
   balance.
2. Find missing values, duplicate rows and incorrect values.
3. Clean a dataset step by step and tell invalid values from plausible outliers.
4. Prepare numerical features and categorical features.
5. Use `LabelEncoder` for the target and `OneHotEncoder` for nominal features,
   and explain the ordinal case.
6. Split data into training, validation and test sets.
7. Explain data leakage and prevent it.
8. Compute min-max scaling, mean normalization and standardization from
   training data only.
9. Read charts of missing values, outliers and scaled features.

## Required imports

All libraries are imported once here. `numpy` builds the data, `pandas`
organizes it, `matplotlib` and `seaborn` draw charts, and `scikit-learn`
splits the data and encodes columns. `np.random.seed` and
`np.random.default_rng(42)` make the generated data reproducible.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

%matplotlib inline

sns.set_theme(style="whitegrid")
np.random.seed(42)
rng = np.random.default_rng(42)

print("Libraries loaded.")

## Dataset introduction

We create a small **customer dataset** in code, one row per customer. No file is
downloaded, so the notebook runs on CPU in a moment.

| Column | Meaning | Unit / values |
|---|---|---|
| `customer_id` | Customer identifier | whole number (not a feature) |
| `age` | Age | years, valid range 0-120 |
| `income_usd` | Yearly income | US dollars, valid if not negative |
| `tenure_months` | Time with the company | months, valid if not negative |
| `plan_type` | Subscription plan | `Basic`, `Standard`, `Premium` (nominal) |
| `satisfaction` | Satisfaction level | `Low`, `Medium`, `High` (ordinal) |
| `churned` | Did the customer leave? | `yes` or `no` (target) |

**Nominal** means categories with no order (`plan_type`); **ordinal** means
categories with a clear order (`satisfaction`). The data is made up for
teaching; it does not describe real customers.

Fixed domain validity rules (used later, no statistics needed):

- `age` must be between 0 and 120 years.
- `income_usd` must not be negative.
- `tenure_months` must not be negative.

## Explore the dataset

Before changing anything, we look at the data: how many rows, which columns,
which types, and how the values are distributed. This first view is the
"before" picture we compare with later.

In [ ]:
# A seeded generator makes the dataset reproducible.
n_customers = 40

customer_id = np.arange(1001, 1001 + n_customers)                  # identifier only
age = rng.integers(19, 68, n_customers)                            # years
income_usd = np.round(rng.normal(52000, 16000, n_customers), -1)   # US dollars per year
tenure_months = rng.integers(1, 73, n_customers)                   # months
plan_type = rng.choice(["Basic", "Standard", "Premium"], n_customers, p=[0.45, 0.35, 0.20])
satisfaction = rng.choice(["Low", "Medium", "High"], n_customers, p=[0.25, 0.45, 0.30])

# The target depends a little on satisfaction, tenure and plan.
churn_prob = (0.25
              + 0.45 * (satisfaction == "Low")
              + 0.20 * (tenure_months < 12)
              - 0.05 * (plan_type == "Premium"))
churned = np.where(rng.random(n_customers) < churn_prob, "yes", "no")

df = pd.DataFrame({
    "customer_id": customer_id,
    "age": age.astype(float),
    "income_usd": income_usd.astype(float),
    "tenure_months": tenure_months,
    "plan_type": plan_type,
    "satisfaction": satisfaction,
    "churned": churned,
})

# Inject realistic data problems, so we can find and fix them.
df.loc[5, "age"] = np.nan             # missing age
df.loc[12, "income_usd"] = np.nan     # missing income
df.loc[20, "plan_type"] = np.nan      # missing category
df.loc[7, "age"] = 250.0              # invalid age (outside 0-120 years)
df.loc[15, "income_usd"] = -8000.0    # invalid income (negative)
df.loc[30, "income_usd"] = 240000.0   # plausible outlier (possible, but rare)
df.loc[25, "churned"] = np.nan        # missing target
df = pd.concat([df, df.iloc[[3, 11]]], ignore_index=True)  # two duplicate rows

print("raw shape (rows, columns):", df.shape)
df.head()

The table has 42 rows and 7 columns. `df.head()` shows the first five customers.
The data already contains missing cells (`NaN`), two impossible values, one
extreme value and two duplicate rows; we find them one by one below.

In [ ]:
print("shape (rows, columns):", df.shape)
print("columns:", list(df.columns))
print("data types:")
print(df.dtypes)

`df.shape` is `(42, 7)`. `customer_id` and `tenure_months` are whole numbers
(`int64`). `age` and `income_usd` are decimal numbers (`float64`) because they
contain missing values. `plan_type`, `satisfaction` and `churned` are text
(`object`).

### Summary statistics

Summary statistics describe a numerical column with a few numbers.

**Small example.** For the values `[10, 20, 30]`:

- count `3`
- mean `(10 + 20 + 30) / 3 = 20`
- median (middle value) `20`
- minimum `10`, maximum `30`
- **population** standard deviation
  `sqrt(((10-20)^2 + (20-20)^2 + (30-20)^2) / 3) = sqrt(66.667) = 8.165`
- **sample** standard deviation
  `sqrt(((10-20)^2 + (20-20)^2 + (30-20)^2) / (3 - 1)) = sqrt(100) = 10.000`

The **mean** is the balance point, the **median** the middle value, and the
**standard deviation** the typical distance from the mean. General forms:

```text
mean           = (1/n) * sum(x_i)
median         = middle value after sorting
population std = sqrt( (1/n) * sum((x_i - mean)^2) )
sample std     = sqrt( (1/(n-1)) * sum((x_i - mean)^2) )
```

Here `x_i` is one value, `n` the count, and `sum` adds over all values. The
sample version divides by `n - 1` and is a little larger. pandas `describe()`
and `.std()` use `ddof=1` (the sample version), so their value is `10.000`
here; the standardization later in this notebook uses the **population** value
`8.165`, matching scikit-learn's `StandardScaler`.

In [ ]:
print("summary statistics of the numerical columns:")
print(df[["age", "income_usd", "tenure_months"]].describe().round(2))

print()
print("plan_type counts:")
print(df["plan_type"].value_counts(dropna=False))
print()
print("satisfaction counts:")
print(df["satisfaction"].value_counts(dropna=False))

The mean age is 49.44 years, but the maximum is 250 years. That impossible value
pulls the mean upward. The income minimum is -8000 dollars, also impossible.
Tenure looks reasonable (3 to 72 months). `customer_id` is left out, because
averaging identifiers is meaningless. The two text columns are summarized by
counts.

### Target distribution

The **target** is the column we want to predict (`churned`). Its
**distribution** is how often each class appears.

**Small example.** In `[no, no, yes, no]`, `no` appears 3 times and `yes` 1
time, so the proportions are `3/4 = 0.75` and `1/4 = 0.25`.

General form:

```text
proportion of a class = count of that class / total count
```

A very imbalanced target (for example 99% `no`) can make accuracy look high
while the model ignores the small class, so we check balance early.

In [ ]:
target_counts = df["churned"].value_counts(dropna=False)
print(target_counts)

labels = target_counts.index.fillna("missing").astype(str)

plt.figure(figsize=(6, 4))
plt.bar(labels, target_counts.values, color="tab:blue")
plt.title("Target Distribution Before Cleaning")
plt.xlabel("churned (target class)")
plt.ylabel("Number of customers")
plt.grid(axis="y")
plt.show()

The raw data has 28 `no`, 13 `yes` and 1 missing target, so about one third of
customers churned. This is workable, though not perfectly balanced. The missing
target row cannot be filled: inventing the answer we want to predict would train
the model on a guess, so that row will be removed.

### Missing values

A **missing value** is an empty cell, shown by pandas as `NaN` ("not a number").

**Small example.** For `[10, NaN, 30]`, `isna()` gives `[False, True, False]`,
and the sum is `1`.

General form: `df.isna()` is `True` for each missing cell, and `.sum()` counts
them per column.

In [ ]:
missing_counts = df.isna().sum()
print(missing_counts)

plt.figure(figsize=(7, 4))
plt.bar(missing_counts.index, missing_counts.values, color="tab:red")
plt.title("Missing Values per Column (Before Cleaning)")
plt.xlabel("Column")
plt.ylabel("Number of missing values")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y")
plt.show()

Only four columns have missing values: `age` (1), `income_usd` (1), `plan_type`
(1) and `churned` (1). The other columns are complete.

### Duplicate rows

A **duplicate row** repeats the same values as another row. Duplicates give one
customer extra weight, so we remove them.

**Small example.** In the table

```text
id  age
1   30
1   30
2   40
```

the first two rows are duplicates, so `duplicated()` marks the second one.

General form: `df.duplicated()` is `True` for a row already seen;
`df.drop_duplicates()` keeps the first copy.

In [ ]:
print("number of duplicate rows:", df.duplicated().sum())
print("the repeated rows:")
print(df[df.duplicated(keep=False)].sort_values("customer_id")[
    ["customer_id", "age", "income_usd", "tenure_months", "plan_type", "satisfaction", "churned"]
])

There are 2 duplicate rows: exact copies of customers 1004 and 1012. They carry
no new information, so they will be removed.

### Incorrect values

An **incorrect value** breaks a fixed rule about what is possible, such as an
age of 250 years or a negative income. We can find these without any statistics,
because the rule is part of the problem definition.

**Small example.** If the rule is "score between 0 and 100", then `-5` and `130`
are incorrect, while `0` and `100` are allowed.

General form: a value is invalid when it fails a fixed test, for example
`not (0 <= age <= 120)`.

In [ ]:
invalid_age = df[df["age"].notna() & ~df["age"].between(0, 120)]
invalid_income = df[df["income_usd"].notna() & (df["income_usd"] < 0)]
print("rows with an age outside 0-120 years:")
print(invalid_age[["customer_id", "age", "income_usd"]])
print()
print("rows with a negative yearly income:")
print(invalid_income[["customer_id", "age", "income_usd"]])

Customer 1008 has an age of 250 years and customer 1016 has a yearly income of
-8000 dollars. Both break a fixed domain rule, so they are incorrect. We do not
guess a value now; we mark them as missing and fill them later using training
data.

## Clean the data step by step

Cleaning uses fixed rules only, so it can happen **before** the split:

1. Remove exact duplicate rows.
2. Replace values that break the domain rules with `NaN`.
3. Remove rows with a missing target.

We do **not** fill missing feature values here, because the filling values
(median and mode) must be learned from the training set only.

In [ ]:
# Step 1: remove exact duplicate rows.
duplicate_count = df.duplicated().sum()
clean_df = df.drop_duplicates().reset_index(drop=True).copy()
print("duplicate rows removed:", duplicate_count)
print("rows after removing duplicates:", len(clean_df))

# Step 2: fixed domain validity rules (no statistics needed).
clean_df.loc[~clean_df["age"].between(0, 120), "age"] = np.nan
clean_df.loc[clean_df["income_usd"] < 0, "income_usd"] = np.nan
clean_df.loc[clean_df["tenure_months"] < 0, "tenure_months"] = np.nan

# Step 3: drop rows whose target is missing (we never invent a target).
clean_df = clean_df[clean_df["churned"].notna()].reset_index(drop=True)
print("rows after dropping the missing target:", len(clean_df))
print("remaining missing values per column:")
print(clean_df.isna().sum())

Two duplicates were removed (42 to 40 rows). The invalid age and income became
`NaN`, so `age` and `income_usd` each have 2 missing values now. Removing the
missing target left 39 rows with a complete target: 26 `no` and 13 `yes`.

### Outliers: invalid values and plausible outliers

An **outlier** is a value far from the others. Two kinds matter:

- An **invalid value** breaks a fixed rule (age 250). It is wrong and is treated
  as missing.
- A **plausible outlier** is possible but rare (a yearly income of 240000
  dollars). It is real data, so we do not delete it.

**Small example.** For `[10, 11, 12, 13, 40]`, the value `40` is far from the
rest. The **interquartile range (IQR)** rule uses the 25% quartile `Q1` and the
75% quartile `Q3`:

- `Q1 = 11`, `Q3 = 13`, so `IQR = 13 - 11 = 2`
- `lower bound = 11 - 1.5 * 2 = 8`
- `upper bound = 13 + 1.5 * 2 = 16`

`40` is above `16`, so it is an outlier; `10, 11, 12, 13` lie inside `[8, 16]`.

General form:

```text
IQR         = Q3 - Q1
lower bound = Q1 - 1.5 * IQR
upper bound = Q3 + 1.5 * IQR
```

What to do with a plausible outlier is a **choice**, not a fixed rule: keep it,
cap it to a bound, or use a robust method (for example the median instead of the
mean). We compare the options on the validation set. Below we demonstrate
**capping**, learned from training data only; a real project may keep a
legitimate extreme.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, column in zip(axes, ["age", "income_usd", "tenure_months"]):
    sns.boxplot(y=clean_df[column], ax=ax, color="tab:blue")
    ax.set_title(column)
    ax.set_ylabel("Value in its own unit")
plt.suptitle("Boxplots of Numerical Features (After Domain Rules)")
plt.tight_layout()
plt.show()

The age box is centred near 45 years, `income_usd` near 53000 dollars, and
`tenure_months` near 40 months. In the middle chart one point near 240000
dollars lies far above the rest: a plausible outlier, because a very high income
is possible. The age chart no longer shows 250 years, because that invalid value
was already changed to missing.

## Split the data: training, validation and test sets

We train a model on one part of the data and check it on data it has never seen.

- **Training set**: the model learns from it, and all statistics (median, mode,
  outlier bounds, scaling values) are learned here.
- **Validation set**: used to compare choices while building the model.
- **Test set**: used once at the end to estimate final performance.

**Small example (data leakage).** Training incomes `[30, 40]`, test income
`[1000]`. If we scale using all three values, the maximum `1000` comes from the
test set, so test information enters training. The correct maximum is the
training maximum `40`.

General form: split first, then learn every number from the training part only.

In [ ]:
X = clean_df.drop(columns=["customer_id", "churned"])
y = clean_df["churned"]
print("feature columns:", list(X.columns))

# 60% training, 20% validation, 20% test, keeping the class balance.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

train_index = X_train.index
val_index = X_val.index
test_index = X_test.index

print("training rows:", len(X_train))
print("validation rows:", len(X_val))
print("test rows:", len(X_test))
print("training target counts:", y_train.value_counts().to_dict())
print("validation target counts:", y_val.value_counts().to_dict())
print("test target counts:", y_test.value_counts().to_dict())

`customer_id` is dropped, so it can never be used as a feature. The 39 rows
become 23 training, 8 validation and 8 test rows. `stratify=y` keeps the same
share of `yes` and `no` in each part: training has 15 `no` and 8 `yes`,
validation 5 and 3, and test 6 and 2. Every part has at least two examples of
each class, which is the minimum for a meaningful check.

## Learn imputation and outlier bounds from the training set only

Now we fill the missing values and cap outliers, using **training** numbers:

- Missing numerical value -> the **training median** of that column.
- Missing category -> the **training mode** (most frequent value).
- Outlier bounds -> the **training** `Q1`, `Q3` and IQR.

We then apply the same numbers to the validation and test sets. Learning them
from the whole dataset would be data leakage.

In [ ]:
numeric_features = ["age", "income_usd", "tenure_months"]

train_median = X_train[numeric_features].median()
train_mode_plan = X_train["plan_type"].mode().iloc[0]
print("training medians:")
print(train_median)
print("training mode of plan_type:", train_mode_plan)

# Fill missing feature values using training numbers.
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()
for frame in (X_train, X_val, X_test):
    for column in numeric_features:
        frame[column] = frame[column].fillna(train_median[column])
    frame["plan_type"] = frame["plan_type"].fillna(train_mode_plan)

print("total missing values in training:", int(X_train.isna().sum().sum()))
print("total missing values in validation:", int(X_val.isna().sum().sum()))
print("total missing values in test:", int(X_test.isna().sum().sum()))

# Compute outlier bounds from the training set only.
outlier_bounds = {}
for column in numeric_features:
    q1 = X_train[column].quantile(0.25)
    q3 = X_train[column].quantile(0.75)
    iqr = q3 - q1
    outlier_bounds[column] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)

print("training outlier bounds:")
for column, (lower, upper) in outlier_bounds.items():
    print(f"  {column}: lower={lower:.2f}, upper={upper:.2f}")

# Optional demonstration: cap values outside the training bounds. Capping is one
# choice; keeping a legitimate extreme or using a robust model are alternatives.
for frame in (X_train, X_val, X_test):
    for column in numeric_features:
        lower, upper = outlier_bounds[column]
        frame[column] = frame[column].clip(lower, upper)

print("largest training income after capping:", X_train["income_usd"].max())

The training medians are 44 years (`age`), 52880 dollars (`income_usd`) and 40
months (`tenure_months`); the most common training plan is `Basic`. All missing
feature values are now filled. The training IQR rule gives an income upper bound
of 84387.5 dollars. As an **optional demonstration**, values above it are
capped: customer 1031's 240000 dollars, which lies in the validation set,
becomes 84387.5, and one high training value (86270) is capped too. The rows
stay. A project could instead keep a legitimate extreme or use a robust model,
and judge that choice on the validation set.

## Prepare categorical features

A model needs numbers, so text categories must be encoded.

- **Label encoding** replaces each category with a whole number. It is fine for
  the **target**, because the target is a single column and the model only needs
  the labels.
- **One-hot encoding** creates one 0/1 column per category. It is the safe
  choice for **nominal features**, because it does not invent an order.
- **Ordinal features** have a real order, so we map them with an explicit,
  meaningful order instead of an arbitrary one.

**Small example.** Colors `["red", "blue", "red"]`:

- Label encoding (alphabetical): `blue -> 0`, `red -> 1`, giving `[1, 0, 1]`.
- One-hot encoding: `red -> [0, 1]` and `blue -> [1, 0]`.

For a size column `Small, Medium, Large`, the ordered map `Small -> 0`,
`Medium -> 1`, `Large -> 2` respects the order. Label encoding could instead
give `Large -> 0`, which would be wrong. That is the **ordinal caveat**.

In [ ]:
# One-hot encode the nominal plan_type using the training categories.
plan_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
plan_encoder.fit(X_train[["plan_type"]])
plan_columns = list(plan_encoder.get_feature_names_out(["plan_type"]))
print("one-hot columns:", plan_columns)

def encode_plan(frame):
    values = plan_encoder.transform(frame[["plan_type"]])
    return pd.DataFrame(values, columns=plan_columns, index=frame.index)

# Ordinal satisfaction: use a meaningful order, not an arbitrary one.
satisfaction_order = {"Low": 0, "Medium": 1, "High": 2}

def build_features(frame):
    return pd.concat([
        frame[numeric_features].reset_index(drop=True),
        frame["satisfaction"].map(satisfaction_order).rename("satisfaction_level").reset_index(drop=True),
        encode_plan(frame).reset_index(drop=True),
    ], axis=1)

X_train_encoded = build_features(X_train)
X_val_encoded = build_features(X_val)
X_test_encoded = build_features(X_test)
print("encoded training shape:", X_train_encoded.shape)
print("encoded training columns:", list(X_train_encoded.columns))
X_train_encoded.head()

# Label encode the target using the training labels only.
target_encoder = LabelEncoder()
target_encoder.fit(y_train)
y_train_encoded = target_encoder.transform(y_train)
y_val_encoded = target_encoder.transform(y_val)
y_test_encoded = target_encoder.transform(y_test)
print("target classes in order:", list(target_encoder.classes_))
print("first training targets (yes/no -> number):", list(zip(y_train[:5], y_train_encoded[:5])))

The one-hot encoder created three plan columns (`plan_type_Basic`,
`plan_type_Premium`, `plan_type_Standard`), and `satisfaction` became the ordered
column `satisfaction_level` (`Low = 0`, `Medium = 1`, `High = 2`). The target
encoder learned `no -> 0` and `yes -> 1` from the training labels only. The
prepared training table has 7 feature columns and no text.

## Feature scaling

Features with very different ranges can make a model learn unevenly. Here `age`
is in tens of years while `income_usd` is in tens of thousands of dollars.
Scaling puts the features on a similar range. Every scaling value is learned
from the training set only.

### Min-max scaling

**Small example.** Take `[10, 20, 30]`. Minimum `10`, maximum `30`, range
`20`. Then:

- `10 -> (10 - 10) / 20 = 0.0`
- `20 -> (20 - 10) / 20 = 0.5`
- `30 -> (30 - 10) / 20 = 1.0`

Every value lands between 0 and 1.

General form:

```text
x_scaled = (x - training_min) / (training_max - training_min)
```

Here `x` is one value, and `training_min` and `training_max` come from the
training column only.

In [ ]:
small_values = np.array([10.0, 20.0, 30.0])
small_min = small_values.min()
small_max = small_values.max()
small_minmax = (small_values - small_min) / (small_max - small_min)
print("small values:", small_values)
print("minimum:", small_min, " maximum:", small_max)
print("min-max scaled:", small_minmax)

The scaled values are `[0.0, 0.5, 1.0]`, exactly as computed by hand.

### Mean normalization

**Small example.** Take the same `[10, 20, 30]`. The mean is
`(10 + 20 + 30) / 3 = 20`, the minimum `10`, the maximum `30`, and the range
`20`. Then:

- `10 -> (10 - 20) / 20 = -0.5`
- `20 -> (20 - 20) / 20 = 0.0`
- `30 -> (30 - 20) / 20 = 0.5`

Values are centred on 0 and usually stay between -1 and 1.

General form:

```text
x_scaled = (x - training_mean) / (training_max - training_min)
```

Here `training_mean` is the mean of the training column.

In [ ]:
small_mean = small_values.mean()
small_meannorm = (small_values - small_mean) / (small_max - small_min)
print("small mean:", small_mean)
print("mean normalized:", small_meannorm)

The mean-normalized values are `[-0.5, 0.0, 0.5]`.

### Standardization

**Small example.** Take `[10, 20, 30]`. The mean is `20` and the population
standard deviation is `sqrt(((10-20)^2 + (20-20)^2 + (30-20)^2) / 3) =
sqrt(200 / 3) = 8.165`. Then:

- `10 -> (10 - 20) / 8.165 = -1.225`
- `20 -> (20 - 20) / 8.165 = 0.0`
- `30 -> (30 - 20) / 8.165 = 1.225`

After standardization a nonconstant training column has mean 0 and standard deviation 1.

General form:

```text
x_scaled = (x - training_mean) / training_std
```

Here `training_std` is the population standard deviation of the training
column. scikit-learn's `StandardScaler` computes the same values.

In [ ]:
small_std = small_values.std(ddof=0)
small_standard = (small_values - small_mean) / small_std
print("small population standard deviation:", round(small_std, 3))
print("standardized:", small_standard)

The population standard deviation is `8.165`, and the standardized values are
about `[-1.225, 0.0, 1.225]`.

### Apply scaling to the prepared data

We compute `training_min`, `training_max`, `training_mean` and `training_std`
from `X_train_encoded`, then apply them to training, validation and test. The
one-hot columns are already 0/1, so we scale only the continuous numerical
features. We keep the standardized version as the final feature matrix.

In [ ]:
train_min = X_train_encoded[numeric_features].min()
train_max = X_train_encoded[numeric_features].max()
train_mean = X_train_encoded[numeric_features].mean()
train_std = X_train_encoded[numeric_features].std(ddof=0)

# A constant feature has zero range and zero standard deviation. Replace a zero
# denominator with 1, so training values become 0 and new values stay finite.
train_range = (train_max - train_min).replace(0, 1)
train_std_safe = train_std.replace(0, 1)

def min_max_scaled(frame):
    return (frame[numeric_features] - train_min) / train_range

def mean_normalized(frame):
    return (frame[numeric_features] - train_mean) / train_range

def standardized(frame):
    return (frame[numeric_features] - train_mean) / train_std_safe

X_train_scaled = X_train_encoded.copy()
X_val_scaled = X_val_encoded.copy()
X_test_scaled = X_test_encoded.copy()
X_train_scaled[numeric_features] = standardized(X_train_encoded)
X_val_scaled[numeric_features] = standardized(X_val_encoded)
X_test_scaled[numeric_features] = standardized(X_test_encoded)

print("training statistics used for scaling:")
print(pd.DataFrame({
    "min": train_min, "max": train_max, "mean": train_mean, "std": train_std,
}).round(2))
print()
print("training mean after standardization:")
print(X_train_scaled[numeric_features].mean().round(6))
print("training standard deviation after standardization:")
print(X_train_scaled[numeric_features].std(ddof=0).round(6))

The training statistics are fixed once: for example, `age` has minimum 23 years,
maximum 64 years, mean 44.30 years and standard deviation 13.11 years. After
standardization nonconstant training columns have mean 0 and standard deviation 1. A
constant column has no spread, so we use a denominator of `1` to keep the
result finite; its training values and standard deviation become 0. Validation
and test use the same training numbers, so their means
are not exactly 0 - that is correct, because those sets are new, unseen data.

In [ ]:
methods = {
    "raw": lambda frame: frame[numeric_features],
    "min-max": min_max_scaled,
    "mean normalized": mean_normalized,
    "standardized": standardized,
}

fig, axes = plt.subplots(len(numeric_features), len(methods), figsize=(15, 9))
for row, column in enumerate(numeric_features):
    for col, (name, transform) in enumerate(methods.items()):
        values = transform(X_train_encoded)[column]
        axes[row, col].hist(values, bins=10, color="tab:blue", edgecolor="white")
        axes[row, col].set_title(f"{column}: {name}")
        axes[row, col].set_xlabel("value")
        axes[row, col].set_ylabel("count")
fig.suptitle("Training Feature Distributions Before and After Scaling")
fig.tight_layout()
plt.show()

The first column shows the raw values: `age` spans tens of years, `income_usd`
tens of thousands of dollars, and `tenure_months` tens of months, so the x-axes
have very different ranges. The min-max column puts every feature between 0 and
1. The mean-normalized column centres each feature near 0 with the same range
width. The standardized column centres each feature at 0 with a spread of about
1. In all columns the shape of each distribution is the same; only the scale
changes.

In [ ]:
# 1. The three splits are disjoint and cover every cleaned row.
assert set(train_index).isdisjoint(val_index)
assert set(train_index).isdisjoint(test_index)
assert set(val_index).isdisjoint(test_index)
assert len(train_index) + len(val_index) + len(test_index) == len(clean_df)

# 2. Scaling statistics were learned from the training set only.
assert np.allclose(
    (X_val_encoded[numeric_features] - train_mean) / train_std_safe,
    X_val_scaled[numeric_features],
)
all_encoded = pd.concat([X_train_encoded, X_val_encoded, X_test_encoded])
all_mean = all_encoded[numeric_features].mean()
assert (train_mean - all_mean).abs().max() > 1e-6

# 3. Transformed arrays are finite, and training columns are standardized.
for matrix in (X_train_scaled, X_val_scaled, X_test_scaled):
    assert np.isfinite(matrix.to_numpy()).all()
assert np.allclose(X_train_scaled[numeric_features].mean(), 0.0, atol=1e-9)
expected_std = (train_std > 0).astype(float)
assert np.allclose(X_train_scaled[numeric_features].std(ddof=0), expected_std, atol=1e-9)

# Constant training values become zero, not a division-by-zero error.
constant_column = pd.Series([5.0, 5.0, 5.0])
constant_std = pd.Series([constant_column.std(ddof=0)]).replace(0, 1).iloc[0]
constant_range = pd.Series([constant_column.max() - constant_column.min()]).replace(0, 1).iloc[0]
assert np.allclose((constant_column - constant_column.mean()) / constant_std, 0)
assert np.allclose((constant_column - constant_column.min()) / constant_range, 0)

# 4. Each split keeps at least two examples of each target class.
for labels in (y_train_encoded, y_val_encoded, y_test_encoded):
    assert len(np.unique(labels)) == 2
    assert np.bincount(labels).min() >= 2

print("All data preparation checks passed.")

All checks passed: the three splits do not overlap and together cover all 39
cleaned rows; the scaling values equal the training statistics and differ from
the full-data mean in this example; every transformed value is finite;
and each split has at least two examples of each class.

## Common mistakes

- **Filling missing values before the split.** The median or mode then uses
  validation and test data, which is data leakage. Learn them from training only.
- **Scaling before the split.** The minimum, maximum, mean and standard
  deviation must come from the training set.
- **Using the mean on a column with outliers.** The median is more robust when
  a few extreme values pull the mean.
- **Deleting or always capping plausible outliers.** They may be real data. Keep
  a legitimate extreme, or cap it only after checking on the validation set;
  robust methods such as the median are another option.
- **Dividing by a zero range or zero standard deviation.** A constant column has
  no spread; replace the zero denominator with `1`.
- **Using label encoding for nominal features.** It invents an order. Use
  one-hot encoding, and use an explicit ordered map only for true ordinal
  features.
- **Keeping `customer_id` as a feature.** An identifier is not a measurement.
- **Imputing the target.** Never invent the answer you want to predict; drop
  rows with a missing target.
- **Ignoring target balance.** Check the class counts before trusting accuracy.

## Summary

In this notebook we prepared a mixed-type customer dataset:

- **Explored** the shape, columns, data types, summary statistics and target
  balance.
- **Found** missing values, 2 duplicate rows and 2 incorrect values.
- **Cleaned** step by step: removed duplicates, applied fixed domain rules,
  and dropped the row with a missing target.
- **Told apart** invalid values (treated as missing) and plausible outliers
  (kept; capping shown as an optional choice).
- **Learned from training only**: medians and a mode for imputation, IQR bounds
  for outliers, and scaling statistics.
- **Encoded** the target with `LabelEncoder`, the nominal `plan_type` with
  `OneHotEncoder`, and the ordinal `satisfaction` with an explicit order.
- **Split** the data into 23 training, 8 validation and 8 test rows with the
  class balance preserved.
- **Scaled** the numerical features with min-max scaling, mean normalization
  and standardization, and checked the result with assertions.